# CertGen CVPR Checkpoint Preflight — Kaggle T4x2

`non_evidence_preflight` · `not_empirical_evidence` · `not paper evidence` · `claim_allowed=false`

Production-hardened, static-validation passed, fixture-runtime passed; real Kaggle preflight is still required. The run contract is hash-bound and supports `resume`, `restart`, and `force_new_run`. GPU work occurs only in isolated subprocess workers.

Select **GPU T4 ×2** in Kaggle before running any code. Kaggle Internet is used only according to the immutable dependency mode; model and extractor loading remains offline from a validated private mount.

This notebook uses multiprocessing `spawn`, keeps CUDA out of the parent, and schedules one subprocess worker per physical GPU.


## 0 Human instructions

Run top-to-bottom only after selecting `GPU T4 ×2`. Never edit a frozen hash in place.


## 1 Immutable user configuration


## 2 Input discovery


In [ ]:
from __future__ import annotations
import hashlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath

MOUNT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/certgen-cvpr")
direct = sorted(MOUNT_ROOT.glob("*/configuration.yaml"))
if len(direct) == 1:
    INPUT_ROOT = direct[0].parent
elif not direct:
    packages = sorted(MOUNT_ROOT.glob("*/*.zip"))
    if len(packages) != 1: raise RuntimeError(f"expected exactly one CertGen input ZIP; found {len(packages)}")
    INPUT_ROOT = Path("/kaggle/working/certgen-input")
    source_hash = hashlib.sha256(packages[0].read_bytes()).hexdigest()
    marker = INPUT_ROOT / ".source_sha256"
    if INPUT_ROOT.exists():
        if not marker.is_file() or marker.read_text(encoding="utf-8").strip() != source_hash: raise FileExistsError("existing extracted input has a different or unverifiable source")
    else:
        partial = INPUT_ROOT.with_name(".certgen-input.partial")
        total, seen = 0, set()
        with zipfile.ZipFile(packages[0]) as archive:
            if archive.testzip() is not None or len(archive.infolist()) > 200000: raise ValueError("input ZIP CRC/member-count failure")
            for info in archive.infolist():
                member = PurePosixPath(info.filename); key = member.as_posix().casefold(); mode = (info.external_attr >> 16) & 0o170000
                if member.is_absolute() or ".." in member.parts or "\\" in info.filename or key in seen or mode == 0o120000: raise ValueError(f"unsafe input ZIP member: {info.filename}")
                if key.endswith((".zip", ".tar", ".tgz", ".tar.gz")): raise ValueError(f"nested archive refused: {info.filename}")
                seen.add(key); total += info.file_size
            if total > 20 * 1024**3: raise ValueError("input ZIP expansion limit exceeded")
            archive.extractall(partial)
        (partial / ".source_sha256").write_text(source_hash + "\n", encoding="utf-8")
        os.replace(partial, INPUT_ROOT)
else:
    raise RuntimeError("multiple direct CertGen configurations found")
sys.path.insert(0, str(INPUT_ROOT))
from certgen.notebooks.kaggle_io import load_frozen_configuration, verify_input_integrity
verify_input_integrity(INPUT_ROOT, ignored={".source_sha256"})
CONFIG = load_frozen_configuration(INPUT_ROOT)


## 3 Environment diagnostics


## 4 Dependency setup and validation

The bootstrap runs `python -m pip check` and writes `dependency_report.json`, `dependency_freeze.txt`, and `pip_check.txt`.


In [ ]:
from certgen.notebooks.environment_bootstrap import bootstrap_environment
ENVIRONMENT = bootstrap_environment(
    "kaggle_t4x2_preflight",
    output_dir=WORK_ROOT / "environment", network_allowed=bool(CONFIG["dependency_network_allowed"]), apply=True, install_mode=CONFIG.get("dependency_mode", "KAGGLE_INTERNET_ON_INSTALL"),
    revalidate_after_restart=bool(os.environ.get("CERTGEN_POST_RESTART")),
)
if ENVIRONMENT["status"] != "ENVIRONMENT_COMPATIBLE":
    raise RuntimeError(ENVIRONMENT["restart_instruction"] or "environment incompatible")

if ENVIRONMENT["pip_check"]["returncode"] != 0:
    raise RuntimeError("python -m pip check failed; inspect pip_check.txt")


## 5 Asset discovery and validation


## 6 Configuration/provenance validation


In [ ]:
MODE = CONFIG["mode"]
if MODE not in {"resume", "restart", "force_new_run"}:
    raise ValueError("mode must be resume, restart, or force_new_run")


In [ ]:
from certgen.notebooks.model_assets import AssetPolicy
from certgen.notebooks.network_policy import network_policy_from_config
ASSET_POLICY = AssetPolicy(CONFIG["asset_policy"])
NETWORK_POLICY = network_policy_from_config(CONFIG)
if ASSET_POLICY is AssetPolicy.ONLINE_PREFLIGHT_DOWNLOAD and not NETWORK_POLICY.model_asset_network_allowed:
    raise RuntimeError("online preflight asset policy requires model asset network")
if CONFIG["kind"] != "preflight" and NETWORK_POLICY.model_asset_network_allowed:
    raise RuntimeError("model asset downloads are confined to checkpoint/extractor preflight")

from certgen.phase1.assets import validate_private_asset_mount
PRIVATE_ASSET_ROOT = Path(CONFIG.get("private_asset_mount", "/kaggle/input/certgen-private-assets"))
REQUIRED_ASSETS = [row["asset_id"] for row in CONFIG.get("assets", [])]
if REQUIRED_ASSETS:
    ASSET_VALIDATION = validate_private_asset_mount(PRIVATE_ASSET_ROOT, REQUIRED_ASSETS)


In [ ]:
from certgen.notebooks.kaggle_io import disk_guard
DISK = disk_guard("/kaggle/working", int(CONFIG.get("required_disk_bytes", 8 * 1024**3)))


In [ ]:
# Parent visibility check deliberately uses nvidia-smi and never imports or initializes PyTorch.
probe = subprocess.run(["nvidia-smi", "-L"], check=True, capture_output=True, text=True)
GPU_LINES = [line for line in probe.stdout.splitlines() if line.strip().startswith("GPU ")]
GPU_COUNT = len(GPU_LINES)
requested = int(CONFIG.get("requested_gpu_count", 2))
if GPU_COUNT < requested and not (GPU_COUNT == 1 and CONFIG.get("allow_single_gpu_fallback") is True):
    raise RuntimeError(f"requested {requested} GPUs but nvidia-smi reported {GPU_COUNT}")


## 7 Tiny dual-GPU dry run


In [ ]:
from certgen.notebooks.run_state import RunIdentity, prepare_run_directory
IDENTITY = RunIdentity(CONFIG["run_id"], CONFIG["configuration_hash"],
                       str(CONFIG.get("source_manifest_hash", CONFIG.get("reference_manifest_hash", "preflight_none"))),
                       str(CONFIG.get("asset_manifest_hash", "preflight_to_be_generated")))
RUN_STATE = prepare_run_directory(WORK_ROOT, IDENTITY, MODE)
RUN_ROOT = Path(RUN_STATE["run_dir"])
frozen_config_copy = RUN_ROOT / "configuration.yaml"
if frozen_config_copy.exists():
    if hashlib.sha256(frozen_config_copy.read_bytes()).hexdigest() != hashlib.sha256((INPUT_ROOT / "configuration.yaml").read_bytes()).hexdigest():
        raise ValueError("run-root frozen configuration differs from the uploaded configuration")
else:
    shutil.copy2(INPUT_ROOT / "configuration.yaml", frozen_config_copy)


In [ ]:
import multiprocessing as mp
mp.set_start_method("spawn", force=True)
from certgen.notebooks.subprocess_orchestrator import WorkerSpec, run_workers
TINY_SPECS = [
    WorkerSpec(
        worker_id=f"tiny_gpu_{gpu}",
        module="certgen.notebooks.workers.diagnostic_worker",
        physical_gpu=gpu,
        shard_id=f"tiny_gpu_{gpu}",
        args=("--out", str(RUN_ROOT / "tiny_gpu_diagnostic" / f"gpu_{gpu}"),
              "--configuration-hash", CONFIG["configuration_hash"],
              "--input-manifest-hash", str(CONFIG.get("input_manifest_hash", CONFIG.get("reference_manifest_hash", "diagnostic_static_input")))),
        completion_marker=str(RUN_ROOT / "tiny_gpu_diagnostic" / f"gpu_{gpu}" / "worker_completion.json"),
        configuration_hash=CONFIG["configuration_hash"],
        input_manifest_hash=str(CONFIG.get("input_manifest_hash", CONFIG.get("reference_manifest_hash", "diagnostic_static_input"))),
        asset_manifest_hash="no_assets_required",
        worker_type="diagnostic",
        config_schema_version="certgen.kaggle.diagnostic_config.v1",
        output_schema_version="certgen.kaggle.diagnostic_output.v1",
    )
    for gpu in range(2)
]
TINY_DUAL_GPU = run_workers(TINY_SPECS, output_dir=RUN_ROOT / "tiny_gpu_orchestration", resume=MODE == "resume")
if sorted(row["physical_gpu"] for row in TINY_DUAL_GPU["workers"]) != [0, 1]:
    raise RuntimeError("tiny dry run did not execute one worker on each physical GPU")


## 8 Runtime calibration


In [ ]:
from certgen.cvpr.contracts import atomic_write_json
CALIBRATION_ROWS = [
    json.loads((RUN_ROOT / "tiny_gpu_diagnostic" / f"gpu_{gpu}" / "diagnostic_report.json").read_text(encoding="utf-8"))
    for gpu in range(2)
]
RUNTIME_CALIBRATION = {
    "model_load_seconds": [row["model_load_seconds"] for row in CALIBRATION_ROWS],
    "warmup_seconds": [row["warmup_seconds"] for row in CALIBRATION_ROWS],
    "throughput_iterations_per_second": [row["throughput_iterations_per_second"] for row in CALIBRATION_ROWS],
    "peak_vram_bytes": [row["peak_allocated_bytes"] for row in CALIBRATION_ROWS],
    "safe_batch_size": min(row["safe_batch_size"] for row in CALIBRATION_ROWS),
    "planning_only": True,
    "not_empirical_evidence": True,
    "claim_allowed": False,
}
atomic_write_json(RUNTIME_CALIBRATION, RUN_ROOT / "runtime_calibration.json")


## 9 Full parallel execution


In [ ]:
from certgen.notebooks.subprocess_orchestrator import WorkerSpec

specs = []
for index, asset in enumerate(CONFIG["assets"]):
    gpu = index % GPU_COUNT
    cache_root = PRIVATE_ASSET_ROOT / asset.get("mount_subdir", asset["model_or_extractor_id"])
    if asset["asset_kind"] == "model":
        model_id = asset["model_or_extractor_id"]
        shard_id = f"model__{model_id}"
        worker_out = RUN_ROOT / "per_model" / model_id
        specs.append(WorkerSpec(
            worker_id=shard_id, module="certgen.notebooks.workers.preflight_worker",
            physical_gpu=gpu, shard_id=shard_id,
            args=("--config", str(INPUT_ROOT / "configuration.yaml"), "--asset-id", asset["asset_id"],
                  "--shard-id", shard_id, "--cache-root", str(cache_root), "--out", str(worker_out)),
            completion_marker=str(worker_out / "worker_completion.json"),
            configuration_hash=CONFIG["configuration_hash"], input_manifest_hash=CONFIG["input_manifest_hash"],
        ))
    else:
        extractor_id = asset["model_or_extractor_id"]
        asset_shard = f"extractor_asset__{extractor_id}"
        asset_out = RUN_ROOT / "per_asset" / extractor_id
        specs.append(WorkerSpec(
            worker_id=asset_shard, module="certgen.notebooks.workers.preflight_worker",
            physical_gpu=gpu, shard_id=asset_shard,
            args=("--config", str(INPUT_ROOT / "configuration.yaml"), "--asset-id", asset["asset_id"],
                  "--shard-id", asset_shard, "--cache-root", str(cache_root), "--out", str(asset_out), "--asset-only"),
            completion_marker=str(asset_out / "worker_completion.json"),
            configuration_hash=CONFIG["configuration_hash"], input_manifest_hash=CONFIG["input_manifest_hash"],
        ))
        extractor_shard = f"extractor__{extractor_id}"
        extractor_out = RUN_ROOT / "per_extractor" / extractor_id
        specs.append(WorkerSpec(
            worker_id=extractor_shard, module="certgen.notebooks.workers.extractor_preflight_worker",
            physical_gpu=gpu, shard_id=extractor_shard,
            args=("--config", str(INPUT_ROOT / "configuration.yaml"), "--extractor-id", extractor_id,
                  "--shard-id", extractor_shard, "--asset-manifest", str(asset_out / "asset_manifest.json"),
                  "--cache-root", str(cache_root), "--out", str(extractor_out)),
            completion_marker=str(extractor_out / "worker_completion.json"),
            configuration_hash=CONFIG["configuration_hash"], input_manifest_hash=CONFIG["input_manifest_hash"],
        ))

from certgen.notebooks.kaggle_io import assert_unique_shards
assert_unique_shards([{"shard_id": spec.shard_id} for spec in specs])


In [ ]:
from certgen.notebooks.subprocess_orchestrator import run_workers
ORCHESTRATION = run_workers(specs, output_dir=RUN_ROOT / "orchestration", timeout_seconds=CONFIG.get("worker_timeout_seconds"), resume=MODE == "resume")


In [ ]:
FAILED = [row for row in ORCHESTRATION["workers"] if row["status"] not in {"COMPLETE", "REUSED_VALID_COMPLETION"}]
for row in ORCHESTRATION["workers"]:
    print(row["worker_id"], row["status"], row.get("log"), row.get("rerun_command"))
if FAILED:
    raise RuntimeError("BLOCKED_PARTIAL_FAILURE; preserve completed shards and use the emitted rerun commands")


## 10 Merge and validation


In [ ]:
from certgen.cvpr.contracts import atomic_write_json
from certgen.notebooks.kaggle_io import all_worker_statuses_complete
if not all_worker_statuses_complete(ORCHESTRATION):
    raise RuntimeError("shard validation failed")
ROOT_STATUS = {"status_code": "PREFLIGHT_PASS", "passed": True, "configuration_hash": CONFIG["configuration_hash"],
                "mode": MODE, "expected_workers": sorted(spec.worker_id for spec in specs),
                "completed_workers": sorted(row["worker_id"] for row in ORCHESTRATION["workers"]),
                "output_schema_version": CONFIG["output_schema_version"],
                "evidence_class": "non_evidence_preflight", "claim_allowed": False}
atomic_write_json(ROOT_STATUS, RUN_ROOT / "status.json")
if CONFIG["kind"] == "preflight":
    ROOT_STATUS["results"] = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(RUN_ROOT.glob("per_model/*/status.json"))]
    ROOT_STATUS["extractor_results"] = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(RUN_ROOT.glob("per_extractor/*/status.json"))]
    atomic_write_json(ROOT_STATUS, RUN_ROOT / "checkpoint_preflight_status.json")
elif CONFIG["kind"] == "generation":
    atomic_write_json(ROOT_STATUS, RUN_ROOT / "generation_status.json")
else:
    atomic_write_json(ROOT_STATUS, RUN_ROOT / "feature_extraction_status.json")


In [ ]:
# Deterministic merge is sample-ID based inside each worker and orchestration status is sorted by worker ID.
MERGE_INDEX = sorted((row["worker_id"], row["shard_id"]) for row in ORCHESTRATION["workers"])
atomic_write_json({"workers": MERGE_INDEX, "configuration_hash": CONFIG["configuration_hash"], "claim_allowed": False}, RUN_ROOT / "merge_index.json")


In [ ]:
from certgen.notebooks.kaggle_io import write_integrity_manifest
INTEGRITY = write_integrity_manifest(RUN_ROOT)


## 11 Atomic output ZIP


In [ ]:
from certgen.notebooks.kaggle_io import copyback_instructions
from certgen.notebooks.final_zip import finalize_output_zip, validate_final_zip, write_multipart_fallback
ZIP_PATH = Path("/kaggle/working") / f"certgen_cvpr_preflight_{CONFIG['run_id']}.zip"
(RUN_ROOT / "copyback_instructions.md").write_text(copyback_instructions("preflight", ZIP_PATH), encoding="utf-8")
write_integrity_manifest(RUN_ROOT)
ZIP = finalize_output_zip(RUN_ROOT, ZIP_PATH, mode=MODE, configuration_hash=CONFIG["configuration_hash"],
                          asset_manifest_hash=str(CONFIG.get("asset_manifest_hash", "preflight_generated")))

if not validate_final_zip(RUN_ROOT, ZIP_PATH)["passed"]:
    raise RuntimeError("final output ZIP revalidation failed")
MULTIPART = write_multipart_fallback(ZIP_PATH) if ZIP_PATH.stat().st_size > 3800 * 1024**2 else None


## 12 Local handoff


## Copy-back and local import

Copy the final ZIP without unpacking or renaming it, preserve its hash, and run:

`python3 -m certgen import preflight <copied-back-zip>`



On failure, preserve completed shards/logs and run only each exact `rerun_command` emitted in the monitoring cell. Changed input, config, or asset hashes require `restart` or `force_new_run`; never reuse incompatible markers.


In [ ]:
FINAL_STATUS = {"status": "RUN_READY_BY_LOCAL_CONTRACT_REAL_KAGGLE_EXECUTION_REQUIRED",
                "output_zip": ZIP, "evidence_class": "non_evidence_preflight", "claim_allowed": False}
print(json.dumps(FINAL_STATUS, indent=2, sort_keys=True))
